# Billboard Hot 100 & Spotify — Exploratory Data Analysis

**Research focus (Q1):** Has the lifespan of Billboard Hot 100 hits changed over time?

This notebook performs exploratory analysis on the two project sources:
1. **Billboard Hot 100** — weekly chart observations, used to calculate hit-song lifespan.
2. **Spotify Top Songs in 73 Countries** — daily chart observations, used for supplementary track-level exploration and a cautious cross-source match.

The primary Q1 outcome is calculated from Billboard only. Spotify begins in October 2023, while the supplied Billboard file ends in November 2021; therefore, the sources do **not** provide overlapping chart periods. A Spotify match must not be interpreted as evidence of a contemporaneous trend.

**Expected processed files**
- `data/processed/billboard_clean.csv`
- `data/processed/billboard_songs.csv`
- `data/processed/spotify_clean.csv`

Run the download and preprocessing scripts before running this notebook. All paths are relative to the repository root.

## 1. Imports and reproducibility

In [ ]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

RANDOM_STATE = 42
DATA_DIR = Path("data/processed")
FIGURE_DIR = Path("visuals/eda")
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid", context="notebook")


## 2. Load the processed datasets

The notebook intentionally reads the processed outputs rather than repeating cleaning decisions. Cleaning, identifier construction, and row-removal counts belong in `scripts/preprocess.py` and the data card.

In [ ]:
BILLBOARD_WEEKLY_PATH = DATA_DIR / "billboard_clean.csv"
BILLBOARD_SONGS_PATH = DATA_DIR / "billboard_songs.csv"
SPOTIFY_PATH = DATA_DIR / "spotify_clean.csv"

required_paths = [BILLBOARD_WEEKLY_PATH, BILLBOARD_SONGS_PATH, SPOTIFY_PATH]
missing_paths = [str(path) for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError(
        "The following processed files are missing: "
        + ", ".join(missing_paths)
        + ". Run the download and preprocessing scripts first."
    )

billboard = pd.read_csv(BILLBOARD_WEEKLY_PATH, parse_dates=["date"])
bb_songs = pd.read_csv(BILLBOARD_SONGS_PATH, parse_dates=["debut_date", "last_date"])
spotify = pd.read_csv(SPOTIFY_PATH)

if "first_seen" in spotify.columns:
    spotify["first_seen"] = pd.to_datetime(spotify["first_seen"], errors="coerce")
if "last_seen" in spotify.columns:
    spotify["last_seen"] = pd.to_datetime(spotify["last_seen"], errors="coerce")
if "album_release_date" in spotify.columns:
    spotify["album_release_date"] = pd.to_datetime(
        spotify["album_release_date"], errors="coerce"
    )

print("Billboard weekly:", billboard.shape)
print("Billboard song-level:", bb_songs.shape)
print("Spotify track-level:", spotify.shape)


## 3. First contact: structure, sample, identifiers, missing values, and duplicates

The main identifiers are:
- Billboard chart observation: `date` + `song_id` (one song in one weekly chart).
- Billboard song-level record: `song_id`.
- Spotify track-level record: `spotify_id` (a track version, not necessarily a unique composition).

The cross-source `song_key` is a normalized text key. It can create false matches or miss legitimate matches, so matching quality must be treated cautiously.

In [ ]:
for name, df in {
    "Billboard weekly": billboard,
    "Billboard song-level": bb_songs,
    "Spotify track-level": spotify,
}.items():
    print(f"\\n===== {name} =====")
    display(df.dtypes.to_frame("dtype"))
    display(df.sample(min(5, len(df)), random_state=RANDOM_STATE))


In [ ]:
def first_contact_report(df, name, key_columns=None):
    print(f"\\n{name}")
    print("-" * len(name))
    missing = df.isna().sum().sort_values(ascending=False)
    missing = missing[missing > 0]
    print("Missing values:")
    display(missing.to_frame("missing_count"))
    print("Exact duplicate rows:", int(df.duplicated().sum()))
    if key_columns:
        print(f"Duplicate key rows ({key_columns}):", int(df.duplicated(key_columns).sum()))

first_contact_report(billboard, "Billboard weekly", ["date", "song_id"])
first_contact_report(bb_songs, "Billboard song-level", ["song_id"])
first_contact_report(spotify, "Spotify track-level", ["spotify_id"])


## 4. Date coverage and data granularity

Billboard is weekly; Spotify is daily. Dates are chart/snapshot dates and are treated as timezone-naive because the source files do not provide a reliable timezone specification.

In [ ]:
date_summary = pd.DataFrame({
    "dataset": ["Billboard weekly", "Billboard song-level", "Spotify track-level"],
    "start_date": [
        billboard["date"].min(),
        bb_songs["debut_date"].min(),
        spotify["first_seen"].min() if "first_seen" in spotify else pd.NaT,
    ],
    "end_date": [
        billboard["date"].max(),
        bb_songs["last_date"].max(),
        spotify["last_seen"].max() if "last_seen" in spotify else pd.NaT,
    ],
    "rows": [len(billboard), len(bb_songs), len(spotify)],
})
display(date_summary)

fig, ax = plt.subplots(figsize=(10, 3.5))
for y, row in date_summary.iterrows():
    if pd.notna(row["start_date"]) and pd.notna(row["end_date"]):
        ax.plot([row["start_date"], row["end_date"]], [y, y], linewidth=8, solid_capstyle="round")
ax.set_yticks(range(len(date_summary)), date_summary["dataset"])
ax.set_title("Coverage windows differ across the two sources")
ax.set_xlabel("Date")
ax.set_ylabel("")
plt.tight_layout()
plt.show()


## 5. Billboard: one-variable exploration

### 5.1 Chart rank and weeks on board

Rank 1 is the highest position. `weeks_on_board` is a source-provided field; Q1 lifespan is independently calculated from distinct weekly chart appearances in the song-level table.

In [ ]:
display(billboard[["rank", "peak_rank", "weeks_on_board"]].describe().T)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(billboard["rank"].dropna(), bins=20, ax=axes[0])
axes[0].set(title="Weekly chart rank distribution", xlabel="Chart rank (1 = highest)", ylabel="Chart observations")
sns.histplot(billboard["weeks_on_board"].dropna(), bins=30, ax=axes[1])
axes[1].set(title="Source-reported weeks on board", xlabel="Weeks on board", ylabel="Chart observations")
plt.tight_layout()
plt.show()


### 5.2 Song-level lifespan distribution

For Q1, `total_weeks` is the number of distinct Billboard chart weeks in which a song appears. The last 365 days of debuting songs are excluded from lifespan comparisons to reduce right-censoring.

In [ ]:
required_song_columns = {"song_id", "debut_date", "total_weeks", "peak"}
missing_song_columns = required_song_columns - set(bb_songs.columns)
if missing_song_columns:
    raise ValueError(f"Billboard song-level file is missing columns: {sorted(missing_song_columns)}")

analysis_end = bb_songs["last_date"].max() if "last_date" in bb_songs else billboard["date"].max()
cutoff_date = analysis_end - pd.Timedelta(days=365)

bb_q1 = bb_songs.loc[bb_songs["debut_date"] <= cutoff_date].copy()
bb_q1["debut_year"] = bb_q1["debut_date"].dt.year
bb_q1["decade"] = (bb_q1["debut_year"] // 10) * 10
bb_q1["short_run"] = bb_q1["total_weeks"] <= 5

print("Analysis end date:", analysis_end.date())
print("Right-censoring cutoff:", cutoff_date.date())
print("Songs before censoring filter:", len(bb_songs))
print("Songs retained for Q1:", len(bb_q1))
print("Songs excluded as recent debuts:", len(bb_songs) - len(bb_q1))

display(bb_q1["total_weeks"].describe(percentiles=[.1, .25, .5, .75, .9, .95]).to_frame())


In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
sns.histplot(bb_q1["total_weeks"], bins=40, ax=ax)
ax.set_xscale("log")
ax.set_title("Billboard hit lifespan is strongly right-skewed")
ax.set_xlabel("Total weeks on chart (log scale)")
ax.set_ylabel("Number of songs")
plt.tight_layout()
plt.show()


## 6. Q1: lifespan and turnover by debut year

The following table contains the main Q1 metrics:
- Median lifespan (`total_weeks`) by debut year.
- Share of songs with a run of five weeks or fewer.
- Number of distinct songs debuting in each year (turnover).

The first and last partial calendar years are excluded from year-to-year comparisons. The final-year exclusion is applied after the right-censoring filter.

In [ ]:
annual = (
    bb_q1.groupby("debut_year")
    .agg(
        songs_debuted=("song_id", "nunique"),
        median_total_weeks=("total_weeks", "median"),
        short_run_share=("short_run", "mean"),
        mean_total_weeks=("total_weeks", "mean"),
        n_songs=("song_id", "nunique"),
    )
    .reset_index()
)

first_full_year = int(bb_q1["debut_year"].min()) + 1
last_full_year = int(bb_q1["debut_year"].max()) - 1
annual_full = annual.loc[
    annual["debut_year"].between(first_full_year, last_full_year)
].copy()

display(annual_full.head(10))
display(annual_full.tail(10))


In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)
axes[0].plot(annual_full["debut_year"], annual_full["median_total_weeks"])
axes[0].set_ylabel("Median weeks")
axes[0].set_title("Median Billboard hit lifespan by debut year")
axes[1].plot(annual_full["debut_year"], annual_full["short_run_share"] * 100)
axes[1].set_ylabel("Songs lasting ≤5 weeks (%)")
axes[1].set_title("Share of short chart runs")
axes[2].bar(annual_full["debut_year"], annual_full["songs_debuted"], width=0.85)
axes[2].set_ylabel("New songs")
axes[2].set_xlabel("Debut year")
axes[2].set_title("Annual Billboard debut count (turnover)")
plt.tight_layout()
plt.show()


## 7. Relationships and group comparisons

These checks explore whether lifespan is associated with peak chart position and whether lifespan distributions differ by decade. Correlation is descriptive and does not establish causation.

In [ ]:
relationship = bb_q1[["total_weeks", "peak"]].dropna()
display(relationship.corr(method="spearman"))

fig, ax = plt.subplots(figsize=(7, 5))
sns.scatterplot(
    data=relationship.sample(min(5000, len(relationship)), random_state=RANDOM_STATE),
    x="peak", y="total_weeks", alpha=0.35, ax=ax
)
ax.set_yscale("log")
ax.set_title("Peak chart position and total chart lifespan")
ax.set_xlabel("Peak rank (1 = highest)")
ax.set_ylabel("Total weeks on chart (log scale)")
plt.tight_layout()
plt.show()


In [ ]:
decade_summary = (
    bb_q1.groupby("decade")
    .agg(
        songs=("song_id", "nunique"),
        median_weeks=("total_weeks", "median"),
        short_run_share=("short_run", "mean"),
        q25=("total_weeks", lambda x: x.quantile(.25)),
        q75=("total_weeks", lambda x: x.quantile(.75)),
    )
    .reset_index()
)
display(decade_summary)

fig, ax = plt.subplots(figsize=(12, 5))
sns.violinplot(data=bb_q1, x="decade", y="total_weeks", inner="quartile", cut=0, ax=ax)
ax.set_yscale("log")
ax.set_title("Billboard hit lifespan distributions by debut decade")
ax.set_xlabel("Debut decade")
ax.set_ylabel("Total weeks on chart (log scale)")
plt.tight_layout()
plt.show()


## 8. Spotify: supplementary exploratory analysis

Spotify is used for a second-source perspective on recent global and country-level chart presence. Its daily Top 50 data is not directly comparable with Billboard's weekly Top 100 lifespan measure. Do not combine `chart_rows` with Billboard `total_weeks` as if they were the same unit.

In [ ]:
spotify_numeric = [
    col for col in [
        "daily_rank", "popularity", "duration_ms",
        "danceability", "energy", "valence", "chart_rows",
        "best_rank", "n_countries"
    ] if col in spotify.columns
]
display(spotify[spotify_numeric].describe().T)

if "n_countries" in spotify.columns:
    fig, ax = plt.subplots(figsize=(8, 4))
    sns.histplot(spotify["n_countries"].dropna(), bins=30, ax=ax)
    ax.set_title("Distribution of countries in which a track charted")
    ax.set_xlabel("Number of countries")
    ax.set_ylabel("Number of tracks")
    plt.tight_layout()
    plt.show()

if {"danceability", "energy", "valence"}.issubset(spotify.columns):
    fig, ax = plt.subplots(figsize=(7, 5))
    sns.heatmap(
        spotify[["danceability", "energy", "valence"]].corr(),
        annot=True, vmin=-1, vmax=1, center=0, cmap="vlag", ax=ax
    )
    ax.set_title("Correlation between selected Spotify audio features")
    plt.tight_layout()
    plt.show()


## 9. Cross-source join audit

The join is based on normalized title and lead-artist text (`song_key`). This is not a globally unique music identifier. The Spotify data window starts after the Billboard data window ends, so matches indicate that a track/title appears in the later Spotify dataset, not that the same chart run continued across both sources.

In [ ]:
if "song_key" in bb_songs.columns and "song_key" in spotify.columns:
    bb_keys = bb_songs["song_key"].dropna().astype(str)
    sp_keys = spotify["song_key"].dropna().astype(str)
    matched_keys = set(bb_keys).intersection(set(sp_keys))
    match_rate = bb_keys.isin(set(sp_keys)).mean()
    print("Unique Billboard song keys:", bb_keys.nunique())
    print("Unique Spotify song keys:", sp_keys.nunique())
    print("Matched unique keys:", len(matched_keys))
    print(f"Billboard song-key match rate: {match_rate:.1%}")
else:
    print("song_key is unavailable in one or both processed datasets.")


## 10. Interpretation checklist and limitations

Before writing the website commentary, use the actual outputs above and verify:

1. Does the median lifespan move consistently, or are there periods of increase and decrease?
2. Does the share of songs lasting five weeks or fewer support the same story as the median?
3. Does annual debut count change, and could this reflect chart-rule changes or industry changes?
4. Are decade distributions heavy-tailed, with a small number of exceptionally long-running hits?
5. Are any apparent breaks aligned with Billboard chart methodology changes?

**Important limitations**
- `song + artist` is a constructed identifier; alternate credits, remixes, reissues, and spelling changes may split or merge identities.
- Seasonal songs can leave and re-enter the chart. Counting distinct chart weeks across the full history measures total chart presence, not necessarily one uninterrupted run.
- The 365-day right-censoring exclusion reduces incomplete lifespans but does not solve all censoring or re-entry issues.
- Billboard methodology changes can create artificial discontinuities. Verify exact effective dates from authoritative Billboard sources before adding annotations to the final trend figure.
- The Spotify and Billboard observation windows do not overlap. The Spotify join is supplementary and potentially biased toward older songs that resurfaced after 2023.
- The notebook's exploratory plots are diagnostic. The two publication-ready Q1 static figures should be generated reproducibly by `scripts/figures.py`, with source notes, readable labels, colour-blind-safe styling, and uncertainty bands where appropriate.